Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGES_PER_SESSION = 5
IMAGE_SIZE = (100, 300)

all_images = []
all_labels = []

# ==== STEP 1: LOAD INDIVIDUAL FINGER IMAGES (Strategy 2) ====
def process_session_strategy2(base_path, session_label):
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Session {session_label}"):
        for finger_id in range(1, NUM_FINGERS + 1):
            folder = f"vein{subject_id:03d}_{finger_id}"
            for img_idx in range(1, IMAGES_PER_SESSION + 1):
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Failed to load: {img_path}")
                    continue
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                all_images.append(img_norm)

                # 🏷️ Enhanced label
                label = f"{session_label}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                all_labels.append(label)
                print(f"🏷️ Label assigned: {label}")

# Process Session 1 and Session 2
process_session_strategy2(base_path_sess1, session_label=1)
process_session_strategy2(base_path_sess2, session_label=2)

all_labels = np.array(all_labels)
print(f"\n✅ Loaded {len(all_images)} images with Strategy 2 (Protocol 1)")

# ==== STEP 2: COMPUTE (2D)²PCA PROJECTION MATRICES ====
def compute_2d2pca_projection(images, num_rows, num_cols):
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n

    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))

    for img in images:
        A = img - mean_img
        G_row += A @ A.T
        G_col += A.T @ A

    G_row /= n
    G_col /= n

    # Eigen decomposition
    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)

    # Sort and select top components
    idx_r = np.argsort(-eig_vals_r)
    idx_c = np.argsort(-eig_vals_c)
    U = eig_vecs_r[:, idx_r[:num_rows]]  # row projection
    V = eig_vecs_c[:, idx_c[:num_cols]]  # column projection

    return U, V

# ==== STEP 3: PROJECT IMAGES USING (2D)²PCA ====
num_row_components = 137
num_col_components = 137
U, V = compute_2d2pca_projection(all_images, num_row_components, num_col_components)

projected_features = []
for img in all_images:
    feat = U.T @ img @ V
    projected_features.append(feat)
    if len(projected_features) <= 3:
        print(f"🧮 Sample shape after (2D)²PCA: {feat.shape}")

# ==== STEP 4: FLATTEN FOR CLASSIFIER ====
flat_features = np.array([f.flatten() for f in projected_features])
print(f"\n✅ Final feature matrix shape: {flat_features.shape}")
print(f"🧾 Number of labels: {len(all_labels)}")


Testing:

In [ ]:
test_data = []
test_labels = []
test_paths = []

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Preparing test data"):
    for img_idx in [6]:  # Protocol 1: last image is test image
        for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder_name, f"{img_idx:02d}.jpg")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Missing: {img_path}")
                    continue

                print(f"✅ Using: {img_path}")
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                test_data.append(img_norm)

                # 🏷️ Enhanced label
                label = f"{session_label}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                test_labels.append(label)
                test_paths.append(img_path)
                print(f"🏷️ Label assigned: {label}")

# Convert to numpy arrays
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# ==== STEP X: PROJECT TEST IMAGES USING (2D)²PCA ====
proj_test_features = [U.T @ img @ V for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"🧾 Number of test samples: {len(test_labels)}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(test_data)

print("\n🔍 Matching test samples (Strategy 2 + Protocol 1 using (2D)²PCA)...\n")

for i in range(total_tests):
    test_img = test_data[i]
    true_label = test_labels[i]

    # Project test sample using (2D)²PCA
    proj_test = U.T @ test_img @ V
    proj_test_flat = proj_test.flatten()

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test_flat), axis=1)

    # Find nearest match
    closest_index = np.argmin(distances)
    predicted_label = all_labels[closest_index]

    # Parse labels: sessionX_subjectYYY_fingerZ_imgWW
    true_parts = true_label.split('_')
    pred_parts = predicted_label.split('_')

    true_person = true_parts[1]  # subjectXXX
    true_finger = true_parts[2]  # fingerY

    pred_person = pred_parts[1]
    pred_finger = pred_parts[2]

    # Check match on person + finger
    if true_person == pred_person and true_finger == pred_finger:
        correct_matches += 1
        result = "✅ CORRECT (person + finger)"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    print(f"{emoji} Test sample {i+1}/{total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Result   : {result}\n")

# Final accuracy
accuracy = (correct_matches / total_tests) * 100
print("📊 Final Results")
print(f"✅ Correct person+finger matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Person + Finger): {accuracy:.2f}%")
